<div align="center">

<a href="https://colab.research.google.com/github/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-12/NB12_time_series_forecasting.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Week 12: Learning from Sequences: Time Series, Forecasting and Recurrent Networks

**Artificial Intelligence Applications in Engineering (MUH-920 Mühendislikte Yapay Zeka Uygulamaları)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)<br>[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

<sub>Content licensed under CC BY 4.0, code under MIT. This course is updated in line with current developments in the field. Last update: October 2026.</sub>

</div>

## About this notebook

The notebook downloads the monthly Mauna Loa carbon dioxide record from NOAA [1], explores its trend and seasonal cycle, and holds out the last eight years as a test period. It compares naive and seasonal naive baselines, a regression on a quadratic trend and Fourier terms, a ridge regression on lag windows with recursive forecasts, and an LSTM trained in PyTorch [2, 3], all on the same test months.

This notebook is the hands-on part of Week 12. It opens with Python step 12: Time-indexed data, baselines and exceptions and continues with the sections listed in the table of contents. The concepts are explained on the [lecture page](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-12/lecture.html) and in the [PDF notes](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-12/Week12_Lecture_Notes.pdf). The [interactive lab](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-12/lab.html) holds the simulation, the self-assessment and the reflection, and the [week overview](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-12/README.md) gives the learning outcomes, the study path, the discipline challenges and the tasks.

## How to use this notebook

Run the cells in order with Shift+Enter. No earlier Python experience is assumed: Python step 12 teaches the Python of the week first, and every further piece of Python appears in a cell marked **Python move**, which explains the syntax right where it is first needed. Exercise cells start with `None` placeholders and print feedback through `check(...)`. Reference solutions sit in collapsed cells: Open them only after a genuine attempt. Cells that download public data need an internet connection. When a source is unreachable, the notebook falls back to a documented synthetic stand-in so that every later cell still runs.

## 0. Setup

The last lines of the setup cell add a hint to every `NameError`. This error appears when a cell uses a name that no cell has defined in the current session, for example because the cells above have not run.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.linear_model import LinearRegression, Ridge

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
torch.manual_seed(0)

def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = abs(float(value) - float(expected)) <= tol * max(1.0, abs(float(expected)))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] {'Correct.' if ok else 'Not yet. Check the logic and run again.'}")

# Hint for a NameError: The usual cause in a notebook is a cell above that has not run in this session.
from IPython import get_ipython


def name_error_hint(shell, etype, value, tb, tb_offset=None):
    """Show a NameError as usual and add the hint."""
    shell.showtraceback((etype, value, tb), tb_offset=tb_offset)
    print("Hint: A name defined in a cell above exists only after that cell has run in this session. "
          "Runtime > Run before (Ctrl+F8) runs all cells above the current one.")


if get_ipython() is not None:
    get_ipython().set_custom_exc((NameError,), name_error_hint)

## Python step 12: Time-indexed data, baselines and exceptions

> **Starting here?** Run the setup cell above first. The first code cell below checks this and stops with a message if the setup has not run in this session.

This step teaches the Python of the week with the week's own example. Read each explanation, run the cell below it and compare the output with the text; then change a value and predict the new result before running the cell again. A quick check and short exercises with immediate feedback close the step. The [Python path](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/PYTHON_PATH.md) shows how the fourteen steps of the course build on each other.

### Dates and a time index

Time series carry their time stamps with them. `pd.date_range` creates a sequence of dates, and a Series with such an index can be selected with date strings, grouped by month or resampled to a coarser time step. The practice series below is synthetic: Two years of daily mean temperatures with a mean of 12 °C, a seasonal swing of 11 °C and random variation from day to day.

In [ ]:
#@title Check that the setup cell has run
if "check" not in globals():
    raise RuntimeError("The setup cell above has not run in this session. "
                       "Run it first, then continue with this step.")

In [ ]:
import numpy as np
import pandas as pd

days = pd.date_range("2024-01-01", "2025-12-31", freq="D")
doy = days.dayofyear.to_numpy()
temps = pd.Series(12 + 11 * np.sin(2 * np.pi * (doy - 110) / 365)
                  + np.random.default_rng(12).normal(0, 2.5, len(days)), index=days, name="temp_C")
print(temps.head(3).round(1))
print(round(float(temps.loc["2025-07"].mean()), 2), "°C mean in July 2025")

The selection `temps.loc["2025-07"]` keeps all days of July 2025; a date string may name a year, a month or a single day.

### Resampling, rolling windows and lags

`resample("MS").mean()` averages each month and labels it with its first day. `rolling(30, center=True).mean()` smooths the series with a moving window of 30 days, and `shift(k)` moves the series by k steps, which creates the lagged inputs used for forecasting. Missing values appear where a shift or a window reaches beyond the data, and `dropna` removes them.

In [ ]:
monthly = temps.resample("MS").mean()
print(monthly.head(3).round(2))
smooth = temps.rolling(30, center=True).mean()
print(int(smooth.isna().sum()), "days without a full window")
lagged = pd.DataFrame({"today": temps, "yesterday": temps.shift(1), "last_year": temps.shift(365)}).dropna()
print(lagged.shape)
print(monthly.groupby(monthly.index.month).mean().round(1).head(3))

The last line groups the monthly means by calendar month, so January 2024 and January 2025 are averaged together, which is the seasonal profile behind the seasonal baselines of this week.

### Baselines on a time-ordered split

A forecast is only useful if it beats simple baselines. For time series the test period must lie after the training period, because shuffled splits would let a model use the future. Three baselines use the year 2024 for training and 2025 for testing. Persistence predicts that tomorrow equals today, the seasonal naive forecast repeats the value 365 days earlier, and the climatology predicts the mean of the same calendar month in the training year.

In [ ]:
def mae(a, b):
    """Mean absolute error of two aligned Series."""
    return float((a - b).abs().mean())


test = temps.loc["2025"]
train = temps.loc["2024"]
persistence = temps.shift(1).loc["2025"]
seasonal_naive = temps.shift(365).loc["2025"]
profile = train.groupby(train.index.month).mean()
climatology = pd.Series(profile.loc[test.index.month].to_numpy(), index=test.index)
for name, fc in [("persistence", persistence), ("seasonal naive", seasonal_naive), ("climatology", climatology)]:
    print(f"{name:15s} MAE {mae(test, fc):.2f} °C")

In this synthetic series the variation from day to day is independent noise, so the monthly profile wins. In real temperature records the weather of one day carries over into the next, and persistence becomes a strong competitor for short horizons. Every forecasting model of this week has to beat the best of such baselines on the same test period.

### When something goes wrong: exceptions

Real data are messy. A logger may write decimal commas, text such as n/a or empty fields, and converting such text with `float` raises a `ValueError`. A `try` block runs code that may fail, and an `except` block handles the failure instead of stopping the program. The notebooks of this course use the same construction around downloads: If a data service cannot be reached, the `except` block switches to a documented synthetic stand-in. A function can also raise an exception itself when it receives input that makes no sense, which stops errors from spreading silently.

In [ ]:
raw = ["12.5", "13,1", "n/a", "", "14.0", "-2.3"]
values, failed = [], 0
for text in raw:
    try:
        values.append(float(text.replace(",", ".")))
    except ValueError:
        failed += 1
print(values, "failed:", failed)


def moving_average(x, window):
    """Mean of every run of `window` consecutive values; window must be at least 1."""
    if window < 1:
        raise ValueError("window must be at least 1")
    return [sum(x[i:i + window]) / window for i in range(len(x) - window + 1)]


print(moving_average(values, 2))
try:
    moving_average(values, 0)
except ValueError as error:
    print("error:", error)

**Quick check.** s is a monthly Series. What does s.shift(1) contain in its first position?

A. The value of the last month  
B. A missing value (NaN)  
C. Zero  
D. The same value as s in its first position

<details><summary>Show the answer</summary>

**B.** Each value moves one step later in time, so nothing is available to fill the first position.

</details>

### Exercises for Python step 12

**Exercise 12.1.** Compute the mean temperature of January 2024 from `temps` and store it in `py_jan_mean`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py12_temps():
    import numpy as np
    import pandas as pd
    d = pd.date_range("2024-01-01", "2025-12-31", freq="D")
    k = d.dayofyear.to_numpy()
    return pd.Series(12 + 11 * np.sin(2 * np.pi * (k - 110) / 365) + np.random.default_rng(12).normal(0, 2.5, len(d)), index=d)


def _py12_jan_mean_reference():
    return float(_py12_temps().loc["2024-01"].mean())

In [ ]:
py_jan_mean = None
check("Exercise 12.1", py_jan_mean, _py12_jan_mean_reference())

**Exercise 12.2.** Count the days of 2025 with a mean temperature above 25 °C and store the count in `py_hot_days`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py12_hot_days_reference():
    return int((_py12_temps().loc["2025"] > 25).sum())

In [ ]:
py_hot_days = None
check("Exercise 12.2", py_hot_days, _py12_hot_days_reference())

**Exercise 12.3.** Convert the entries of `["7,5", "8.25", "x", "9"]` to numbers with `try` and `except`, skip the entries that fail, and store the sum of the valid numbers in `py_valid_sum`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py12_valid_sum_reference():
    total = 0.0
    for s in ["7,5", "8.25", "x", "9"]:
        try:
            total += float(s.replace(",", "."))
        except ValueError:
            pass
    return total

In [ ]:
py_valid_sum = None
check("Exercise 12.3", py_valid_sum, _py12_valid_sum_reference())

**Exercise 12.4.** Store the MAE of the persistence forecast for 2025 in `py_mae_persist`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py12_mae_persist_reference():
    s = _py12_temps()
    return float((s.loc["2025"] - s.shift(1).loc["2025"]).abs().mean())

In [ ]:
py_mae_persist = None
check("Exercise 12.4", py_mae_persist, _py12_mae_persist_reference())

**Exercise 12.5.** Store the name of the baseline with the lowest MAE, exactly as printed above, in `py_best_baseline`. Use `min` with a `key` rather than reading the value off the output.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py12_best_baseline_reference():
    import pandas as pd
    s = _py12_temps()
    te, tr = s.loc["2025"], s.loc["2024"]
    prof = tr.groupby(tr.index.month).mean()
    fc = {"persistence": s.shift(1).loc["2025"], "seasonal naive": s.shift(365).loc["2025"],
          "climatology": pd.Series(prof.loc[te.index.month].to_numpy(), index=te.index)}
    return min(fc, key=lambda k: float((te - fc[k]).abs().mean()))

In [ ]:
py_best_baseline = None
check("Exercise 12.5", py_best_baseline, _py12_best_baseline_reference())

## 1. The Mauna Loa record

> **Starting here?** Run the setup cell at the top of the notebook first, or choose **Runtime > Run before** (Ctrl+F8). The first code cell below checks this and stops with a message if the setup has not run in this session.

NOAA's Global Monitoring Laboratory publishes the monthly mean carbon dioxide mole fraction at Mauna Loa as a CSV file whose header lines start with `#` [1]. Missing months in the early record are interpolated by NOAA. The fallback generates a series with an accelerating trend and an annual cycle of similar size; its values are illustrative.

**Python move: comment lines and dates.** `comment="#"` skips the header lines, and `pd.to_datetime` builds dates from year and month columns.

In [ ]:
#@title Check that the setup cell has run
if "check" not in globals():
    raise RuntimeError("The setup cell at the top of the notebook has not run in this session. "
                       "Run it first, or choose Runtime > Run before (Ctrl+F8), then continue with this section.")

In [ ]:
URL = "https://gml.noaa.gov/webdata/ccgg/trends/co2/co2_mm_mlo.csv"
try:
    raw = pd.read_csv(URL, comment="#")
    value = next(c for c in raw.columns if c.strip().lower() == "average")
    co2 = pd.Series(raw[value].to_numpy(float), index=pd.to_datetime(dict(year=raw["year"], month=raw["month"], day=1)), name="co2")
    CO2_SOURCE = "NOAA GML, Mauna Loa monthly means"
except Exception as error:
    print("Download failed, using a synthetic stand-in:", type(error).__name__)
    idx = pd.date_range("1958-03-01", "2026-06-01", freq="MS")
    t = (np.arange(len(idx)) + 2) / 12
    rng = np.random.default_rng(1958)
    co2 = pd.Series(315 + 0.8 * t + 0.0125 * t ** 2 + 3.0 * np.sin(2 * np.pi * (idx.month - 2) / 12) + 0.8 * np.cos(4 * np.pi * idx.month / 12)
                    + 0.3 * rng.normal(size=len(idx)), index=idx, name="co2")
    CO2_SOURCE = "synthetic stand-in"
co2 = co2[co2 > 0]
print(CO2_SOURCE, co2.index[0].date(), "to", co2.index[-1].date(), len(co2), "months")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].plot(co2); ax[0].set_ylabel("CO2 (ppm)"); ax[0].set_title("Monthly means")
detrended = co2 - co2.rolling(13, center=True).mean()
detrended.groupby(detrended.index.month).mean().plot(ax=ax[1], marker="o"); ax[1].set_xlabel("month"); ax[1].set_title("Average seasonal cycle")
plt.tight_layout(); plt.show()

## 2. A chronological split and two baselines

> **Starting here?** This section uses results of the cells above it, so choose **Runtime > Run before** (Ctrl+F8) first. The first code cell below checks this and stops with a message if they have not run in this session.

The last 96 months form the test period. The naive forecast repeats the last training value; the seasonal naive forecast repeats the last training year [4].

In [ ]:
#@title Check that the cells above have run
if not all(name in globals() for name in ["check", "co2"]):
    raise RuntimeError("This section uses results of the cells above, which have not run in this session. "
                       "Choose Runtime > Run before (Ctrl+F8), then continue with this section.")

In [ ]:
H = 96
train, test = co2.iloc[:-H], co2.iloc[-H:]
y_tr, y_te = train.to_numpy(), test.to_numpy()
mae = lambda a, b: float(np.mean(np.abs(np.asarray(a) - np.asarray(b))))
naive = np.repeat(y_tr[-1], H)
snaive = np.array([y_tr[len(y_tr) - 12 + (k % 12)] for k in range(H)])
print(f"naive MAE {mae(naive, y_te):.2f} ppm, seasonal naive MAE {mae(snaive, y_te):.2f} ppm")

### Your turn, exercise 1

The seasonal naive forecast ignores the trend. Add a drift: For each forecast month, add the average yearly increase of the last ten training years multiplied by the number of years ahead, `(k // 12 + 1)` for step k. Store the test MAE of this "seasonal naive with drift" forecast in `mae_drift`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _mae_drift_reference():
    yearly = (y_tr[-1] - y_tr[-121]) / 10
    f = np.array([y_tr[len(y_tr) - 12 + (k % 12)] + yearly * (k // 12 + 1) for k in range(H)])
    return mae(f, y_te)

In [ ]:
mae_drift = None   # your computation
check("Exercise 1", mae_drift, _mae_drift_reference(), tol=1e-6)

## 3. Regression on trend and Fourier terms

> **Starting here?** This section uses results of the cells above it, so choose **Runtime > Run before** (Ctrl+F8) first. The first code cell below checks this and stops with a message if they have not run in this session.

**Python move: building a design matrix from time.** The features are time, time squared and two sine-cosine pairs with a period of twelve months.

In [ ]:
#@title Check that the cells above have run
if not all(name in globals() for name in ["check", "co2", "mae", "snaive", "test", "train", "y_te", "y_tr"]):
    raise RuntimeError("This section uses results of the cells above, which have not run in this session. "
                       "Choose Runtime > Run before (Ctrl+F8), then continue with this section.")

In [ ]:
def design(index, t0=co2.index[0]):
    t = ((index.year - t0.year) * 12 + (index.month - t0.month)).to_numpy() / 120.0
    m = index.month.to_numpy()
    return np.column_stack([t, t ** 2, np.sin(2 * np.pi * m / 12), np.cos(2 * np.pi * m / 12), np.sin(4 * np.pi * m / 12), np.cos(4 * np.pi * m / 12)])


reg = LinearRegression().fit(design(train.index), y_tr)
fourier = reg.predict(design(test.index))
print(f"trend + Fourier regression MAE {mae(fourier, y_te):.2f} ppm")
plt.figure(figsize=(9, 3.4))
plt.plot(co2.iloc[-240:], color="gray", label="observed")
plt.plot(test.index, snaive, label="seasonal naive"); plt.plot(test.index, fourier, label="trend + Fourier")
plt.ylabel("CO2 (ppm)"); plt.legend(); plt.title("Forecasts for the test period"); plt.show()

## 4. Lag windows and recursive forecasts

> **Starting here?** This section uses results of the cells above it, so choose **Runtime > Run before** (Ctrl+F8) first. The first code cell below checks this and stops with a message if they have not run in this session.

The series is first differenced, because changes from month to month are closer to stationary than the level itself. Windows of 24 differences predict the next difference with ridge regression; for the test period, each forecast difference is fed back as input, and the differences are summed to levels.

### Your turn, exercise 2

How many windows of length 25 (24 inputs and 1 target) does `sliding_window_view` create from the training differences `d_tr`? Store the number in `n_windows`.

In [ ]:
#@title Check that the cells above have run
if not all(name in globals() for name in ["check", "H", "mae", "y_te", "y_tr"]):
    raise RuntimeError("This section uses results of the cells above, which have not run in this session. "
                       "Choose Runtime > Run before (Ctrl+F8), then continue with this section.")

In [ ]:
d_tr = np.diff(y_tr)
L = 24

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _n_windows_reference():
    return len(d_tr) - (L + 1) + 1

In [ ]:
n_windows = None   # your computation
check("Exercise 2", n_windows, _n_windows_reference())

In [ ]:
W = sliding_window_view(d_tr, L + 1)
ridge = Ridge(alpha=1.0).fit(W[:, :L], W[:, L])


def recursive(model_step, history, steps):
    hist, out = list(history), []
    for _ in range(steps):
        nxt = model_step(np.array(hist[-L:]))
        out.append(nxt); hist.append(nxt)
    return np.array(out)


d_fc = recursive(lambda x: float(ridge.predict(x[None, :])[0]), d_tr, H)
lag_fc = y_tr[-1] + np.cumsum(d_fc)
print(f"ridge on lag windows MAE {mae(lag_fc, y_te):.2f} ppm")

## 5. An LSTM

> **Starting here?** This section uses results of the cells above it, so choose **Runtime > Run before** (Ctrl+F8) first. The first code cell below checks this and stops with a message if they have not run in this session.

**Python move: nn.LSTM with batch_first.** The input has shape (batch, time, features): here windows of 24 standardised differences with one feature. The last hidden state feeds a linear layer that predicts the next difference [2].

In [ ]:
#@title Check that the cells above have run
if not all(name in globals() for name in ["check", "d_tr", "H", "L", "mae", "recursive", "W", "y_te", "y_tr"]):
    raise RuntimeError("This section uses results of the cells above, which have not run in this session. "
                       "Choose Runtime > Run before (Ctrl+F8), then continue with this section.")

In [ ]:
mu, sd = d_tr.mean(), d_tr.std()
Wz = (W - mu) / sd
Xt = torch.tensor(Wz[:, :L, None], dtype=torch.float32); yt = torch.tensor(Wz[:, L:], dtype=torch.float32)


class Forecaster(nn.Module):
    def __init__(self, hidden=32):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden, batch_first=True)
        self.out = nn.Linear(hidden, 1)

    def forward(self, x):
        seq, _ = self.lstm(x)
        return self.out(seq[:, -1])


lstm = Forecaster()
opt = torch.optim.Adam(lstm.parameters(), lr=3e-3)
for epoch in range(60):
    perm = torch.randperm(len(Xt))
    for i in range(0, len(Xt), 64):
        b = perm[i:i + 64]
        opt.zero_grad(); loss = nn.functional.mse_loss(lstm(Xt[b]), yt[b]); loss.backward(); opt.step()
lstm.eval()
with torch.no_grad():
    step = lambda x: float(lstm(torch.tensor(((x - mu) / sd)[None, :, None], dtype=torch.float32))[0, 0]) * sd + mu
    lstm_fc = y_tr[-1] + np.cumsum(recursive(step, d_tr, H))
print(f"LSTM MAE {mae(lstm_fc, y_te):.2f} ppm")

## 6. All methods on the same test months

> **Starting here?** This section uses results of the cells above it, so choose **Runtime > Run before** (Ctrl+F8) first. The first code cell below checks this and stops with a message if they have not run in this session.

In [ ]:
#@title Check that the cells above have run
if not all(name in globals() for name in ["check", "design", "fourier", "lag_fc", "lstm_fc", "mae", "naive", "snaive", "test", "train",
                                         "y_te", "y_tr"]):
    raise RuntimeError("This section uses results of the cells above, which have not run in this session. "
                       "Choose Runtime > Run before (Ctrl+F8), then continue with this section.")

In [ ]:
results = {"naive": naive, "seasonal naive": snaive, "trend + Fourier regression": fourier, "ridge on lag windows": lag_fc, "LSTM": lstm_fc}
table = pd.Series({k: mae(v, y_te) for k, v in results.items()}, name="test MAE (ppm)").round(3)
plt.figure(figsize=(9, 3.4))
plt.plot(test.index, y_te, color="k", lw=2, label="observed")
for k in ["trend + Fourier regression", "ridge on lag windows", "LSTM"]:
    plt.plot(test.index, results[k], label=k)
plt.ylabel("CO2 (ppm)"); plt.legend(fontsize=8); plt.show()
table

### Your turn, exercise 3

Refit the trend and Fourier regression without the squared time term, that is, with a linear trend only, and store its test MAE in `mae_linear_trend`. Why does the error grow over the test period?

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _mae_linear_trend_reference():
    cols = [0, 2, 3, 4, 5]
    m = LinearRegression().fit(design(train.index)[:, cols], y_tr)
    return mae(m.predict(design(test.index)[:, cols]), y_te)

In [ ]:
mae_linear_trend = None   # your computation
check("Exercise 3", mae_linear_trend, _mae_linear_trend_reference(), tol=1e-6)

## Next steps

Take the [self-assessment](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-12/lab.html#check) in the interactive lab, then answer the [reflection prompts](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-12/lab.html#reflect) and export the learning log. The [discipline challenges](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-12/README.md#discipline-challenges), the [weekly task](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-12/README.md#weekly-task) and the [research assignment](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-12/README.md#research-and-report-assignment-optional) are listed in the week overview, and [Week 13](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-13/README.md) starts with its overview.

## References

[1] Lan, X., & Keeling, R. (2026). *Trends in atmospheric carbon dioxide: Mauna Loa CO2 monthly mean data. NOAA Global Monitoring Laboratory and Scripps Institution of Oceanography*. <https://gml.noaa.gov/ccgg/trends/>

[2] Hochreiter, S., & Schmidhuber, J. (1997). Long short-term memory. *Neural Computation*, *9*(8), 1735-1780. <https://doi.org/10.1162/neco.1997.9.8.1735>

[3] Paszke, A., Gross, S., Massa, F., et al. (2019). PyTorch: An imperative style, high-performance deep learning library. In *Advances in Neural Information Processing Systems 32* (pp. 8024-8035). <https://arxiv.org/abs/1912.01703>

[4] Hyndman, R. J., & Athanasopoulos, G. (2021). *Forecasting: Principles and Practice* (3rd ed.). OTexts. <https://otexts.com/fpp3/>